# Lab 07 — AI_REDACT & AI_PARSE_DOCUMENT

**Redact** masks PII automatically. **Parse Document** extracts text from PDFs and images.

| Function | Returns | Use Case |
|---|---|---|
| `AI_REDACT` | Text with PII masked | GDPR/CCPA compliance, data sharing |
| `AI_PARSE_DOCUMENT` | Extracted text/structure from files | Invoice processing, contract analysis |


## Step 1 — Environment Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — AI_REDACT: Automatic PII Detection

In [ ]:
CREATE OR REPLACE TABLE pii_records (
    record_id INT AUTOINCREMENT, record_type VARCHAR(50), raw_text TEXT
);

INSERT INTO pii_records (record_type, raw_text) VALUES
('support_ticket', 'Hi, my name is Sarah Johnson and my account number is ACC-789456. I can be reached at sarah.johnson@email.com or 555-0142. My shipping address is 742 Evergreen Terrace, Springfield, IL 62704.'),
('medical_note', 'Patient: Robert Chen, DOB: 03/15/1985, SSN: 321-54-9876. Presented with symptoms of seasonal allergies. Prescribed cetirizine 10mg daily.'),
('financial_record', 'Transfer from James Williams (account 4532-1234-5678-9012) to Maria Garcia (account 5678-9012-3456-7890). Amount: $15,750.00.'),
('employee_record', 'New hire: Emily Davis, Employee ID: EMP-2024-0892. Personal email: emily.d@gmail.com. Phone: (408) 555-0198.'),
('customer_chat', 'My username is jdoe2024 and my registered phone is 212-555-0167 and the email on file should be john.doe@company.com.');

In [ ]:
SELECT
    record_type,
    SNOWFLAKE.CORTEX.AI_REDACT(raw_text) AS redacted_text
FROM pii_records;

In [ ]:
SELECT
    record_type,
    LEFT(raw_text, 100) AS original,
    LEFT(SNOWFLAKE.CORTEX.AI_REDACT(raw_text)::STRING, 100) AS redacted
FROM pii_records;

## Step 3 — AI_PARSE_DOCUMENT: Extract from PDFs

`AI_PARSE_DOCUMENT` extracts text from documents on a Snowflake stage.
Two modes: `OCR` (raw text) and `LAYOUT` (preserves structure).

> **Note:** This requires PDF files on the `@invoice_pdfs` stage. Upload via Snowsight or PUT.


In [ ]:
CREATE STAGE IF NOT EXISTS invoice_pdfs
    COMMENT = 'Invoice PDF files for AI_PARSE_DOCUMENT demo';

In [ ]:
LIST @invoice_pdfs;

In [ ]:
SELECT
    RELATIVE_PATH AS file_name,
    SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
        TO_FILE(@invoice_pdfs, RELATIVE_PATH),
        {'mode': 'OCR'}
    ) AS parsed_content
FROM DIRECTORY(@invoice_pdfs)
LIMIT 3;

In [ ]:
SELECT
    RELATIVE_PATH AS file_name,
    SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
        TO_FILE(@invoice_pdfs, RELATIVE_PATH),
        {'mode': 'LAYOUT'}
    ) AS parsed_content
FROM DIRECTORY(@invoice_pdfs)
LIMIT 3;

## Step 4 — Chain: Parse → Extract with COMPLETE

In [ ]:
WITH parsed AS (
    SELECT RELATIVE_PATH AS file_name,
        SNOWFLAKE.CORTEX.AI_PARSE_DOCUMENT(
            TO_FILE(@invoice_pdfs, RELATIVE_PATH), {'mode': 'LAYOUT'}
        ):content::STRING AS doc_text
    FROM DIRECTORY(@invoice_pdfs)
)
SELECT file_name,
    SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5',
        'Extract from this invoice and return JSON: {invoice_number, date, vendor_name, total_amount}. Document: ' || doc_text
    ) AS extracted_fields
FROM parsed;

## Key Takeaways

- `AI_REDACT` detects names, SSNs, emails, phones, addresses — no configuration needed.
- `AI_PARSE_DOCUMENT` supports `OCR` (raw text) and `LAYOUT` (structure-preserving) modes.
- Chain `PARSE_DOCUMENT` → `COMPLETE` to extract specific fields from any document.
- Essential for compliance (GDPR/CCPA), data sharing, and document processing pipelines.
